## 02 - Preprocessing

Loads the cleaned dataset, encodes both target labels, splits into train/test,
applies TF-IDF vectorization, and saves all outputs to data/processed/.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp
import joblib
import json
import os

### 1. Load Data

In [2]:
df = pd.read_csv("../data/processed/medquad_model_ready.csv")
df = df.drop(columns=["focus_area"])

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape: (14964, 3)
Columns: ['question_clean', 'qtype', 'department']


,question_clean,qtype,department
0,what is glaucoma,definition,Ophthalmology
1,what causes glaucoma,causes,Ophthalmology
2,what are the symptoms of glaucoma,symptoms,Ophthalmology
3,what are the treatments for glaucoma,treatment,Ophthalmology
4,who is at risk for glaucoma,definition,Ophthalmology


### 2. Encode Labels

Both targets are string labels. We use LabelEncoder on each.
Mappings are saved as JSON so predictions can be decoded later.

In [3]:
qtype_enc = LabelEncoder()
dept_enc  = LabelEncoder()

df["qtype_encoded"] = qtype_enc.fit_transform(df["qtype"])
df["dept_encoded"]  = dept_enc.fit_transform(df["department"])

print("qtype classes :", list(qtype_enc.classes_))
print("department classes:", list(dept_enc.classes_))

qtype classes : ['causes', 'definition', 'diagnosis', 'epidemiology', 'genetic', 'prevention', 'prognosis', 'symptoms', 'treatment']
department classes: ['Cardiology', 'Dermatology', 'ENT', 'Endocrinology & Metabolism', 'Gastroenterology & Hepatology', 'General Medicine & Public Health', 'Genetics & Rare Diseases', 'Geriatrics & Aging', 'Gynecology & Obstetrics', 'Hematology', 'Immunology & Allergy', 'Infectious Disease', 'Nephrology & Urology', 'Neurology', 'Oncology', 'Ophthalmology', 'Orthopedics & Rheumatology', 'Pediatrics', 'Psychiatry & Mental Health', 'Pulmonology']


In [10]:
os.makedirs("../models", exist_ok=True)
qtype_map = {int(i): c for i, c in enumerate(qtype_enc.classes_)}
dept_map  = {int(i): c for i, c in enumerate(dept_enc.classes_)}

os.makedirs("../data/processed", exist_ok=True)

with open("../models/qtype_mapping.json", "w") as f:
    json.dump(qtype_map, f, indent=2)

with open("../models/dept_mapping.json", "w") as f:
    json.dump(dept_map, f, indent=2)

print("Label mappings saved.")

Label mappings saved.


### 3. Train / Test Split

80/20 split. Stratified on qtype because it has the most class imbalance
(definition has ~4600 samples, prevention has ~186).

In [8]:
X = df["question_clean"]
y_qtype = df["qtype_encoded"]
y_dept  = df["dept_encoded"]

X_train, X_test, yq_train, yq_test, yd_train, yd_test = train_test_split(
    X, y_qtype, y_dept,
    test_size=0.2,
    random_state=42,
    stratify=y_qtype
)

print(f"Train: {len(X_train)}  |  Test: {len(X_test)}")

Train: 11971  |  Test: 2993


In [9]:
train_dist = pd.Series(yq_train.values).value_counts(normalize=True).sort_index()
test_dist  = pd.Series(yq_test.values).value_counts(normalize=True).sort_index()

check = pd.DataFrame({"train %": train_dist, "test %": test_dist})
check.index = qtype_enc.classes_
check = (check * 100).round(2)
print(check)

              train %  test %
causes           4.39    4.41
definition      31.01   31.01
diagnosis        4.10    4.11
epidemiology     7.44    7.45
genetic         16.59   16.61
prevention       1.24    1.24
prognosis        2.36    2.34
symptoms        17.94   17.94
treatment       14.92   14.90


### 4. TF-IDF Vectorization

Fit only on train, then transform both sets.

Parameters chosen:
- max_features=10000   : keeps the top 10k terms, enough for 15k short medical questions
- ngram_range=(1,2)    : unigrams + bigrams capture patterns like "what causes", "how is"
- min_df=2             : drops terms that appear in only 1 document (noise)
- max_df=0.95          : drops terms that appear in 95%+ of documents (too common to help)
- sublinear_tf=True    : log-scales term frequency, works better with Naive Bayes

In [11]:
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    strip_accents="unicode",
    analyzer="word"
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print("Train matrix:", X_train_tfidf.shape)
print("Test matrix :", X_test_tfidf.shape)
print("Sparsity    : {:.2f}%".format(
    100 * (1 - X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1]))
))

Train matrix: (11971, 8607)
Test matrix : (2993, 8607)
Sparsity    : 99.86%


In [12]:
feature_names = tfidf.get_feature_names_out()
print("Sample features:", feature_names[:10].tolist())
print("Sample bigrams :", [f for f in feature_names if " " in f][:10])

Sample features: ['10', '11', '11betahydroxylase', '11betahydroxylase deficiency', '12', '13', '14', '14 syndrome', '15', '15 syndrome']
Sample bigrams : ['11betahydroxylase deficiency', '14 syndrome', '15 syndrome', '15q112 microdeletion', '15q133 microdeletion', '15q24 microdeletion', '16p112 deletion', '16p133 deletion', '16q243 microdeletion', '17 alphahydroxylase1720lyase']


### 5. Save Outputs

In [14]:
os.makedirs("../models", exist_ok=True)
# Sparse matrices
sp.save_npz("../data/processed/X_train_tfidf.npz", X_train_tfidf)
sp.save_npz("../data/processed/X_test_tfidf.npz",  X_test_tfidf)

# Labels (both targets, both splits)
np.save("../data/processed/yq_train.npy", yq_train.values)
np.save("../data/processed/yq_test.npy",  yq_test.values)
np.save("../data/processed/yd_train.npy", yd_train.values)
np.save("../data/processed/yd_test.npy",  yd_test.values)

# Vectorizer and encoders
joblib.dump(tfidf,     "../models/tfidf_vectorizer.pkl")
joblib.dump(qtype_enc, "../models/qtype_encoder.pkl")
joblib.dump(dept_enc,  "../models/dept_encoder.pkl")

print("Saved files:")
for f in sorted(os.listdir("../data/processed/")):
    print(" ", f)

Saved files:
  X_test_tfidf.npz
  X_train_tfidf.npz
  medquad_model_ready.csv
  yd_test.npy
  yd_train.npy
  yq_test.npy
  yq_train.npy
